# Day 3 — Agentic AI Meeting Preparation Assistant

## Objective

Build an Agentic AI Meeting Preparation Assistant that helps a user prepare for an upcoming client meeting.

The system will gather information from client documents, previous meeting notes, and stored memories, then generate a concise meeting brief.

The project demonstrates:

- Retrieval-Augmented Generation (RAG)
- Vector database search using FAISS
- Short-term memory
- Long-term memory
- Agentic workflow
- Tool usage
- Meeting brief generation

In [2]:
import os
import numpy as np
import faiss

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from google import genai

print("NumPy:", np.__version__)
print("FAISS:", faiss.__version__)

NumPy: 2.5.2
FAISS: 1.15.0


In [4]:
import os

SYSTEM_CA = "/etc/ssl/certs/ca-certificates.crt"

os.environ["REQUESTS_CA_BUNDLE"] = SYSTEM_CA
os.environ["SSL_CERT_FILE"] = SYSTEM_CA
os.environ["CURL_CA_BUNDLE"] = SYSTEM_CA

print("Using certificate bundle:", SYSTEM_CA)

Using certificate bundle: /etc/ssl/certs/ca-certificates.crt


In [5]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [6]:
test_text = "Acme Corp is an important client."

test_embedding = embedding_model.encode([test_text])

print("Embedding shape:", test_embedding.shape)

Embedding shape: (1, 384)


In [7]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", bool(api_key))

API key loaded: True


In [8]:
client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


## Step 2 — Create Sample Client Data

A fictional client, Acme Corp, is used to demonstrate the meeting preparation assistant.

The dataset contains client information, previous meeting notes, and project documentation. These documents will later be indexed and retrieved using RAG.

In [9]:
from pathlib import Path

data_path = Path("data")

print("Files in data folder:")

for file in data_path.iterdir():
    print("-", file.name)

Files in data folder:
- company_documents.txt
- client_info.txt
- meeting_notes.txt


In [10]:
documents = {}

for file in data_path.glob("*.txt"):
    documents[file.name] = file.read_text(encoding="utf-8")

for filename, content in documents.items():
    print(f"\n{'=' * 60}")
    print(filename)
    print('=' * 60)
    print(content[:500])


company_documents.txt
Acme Corp — Project Documentation

Project: Online Ordering Platform Modernization

Project Overview:

The project aims to modernize Acme Corp's online ordering platform by improving system performance, API reliability, and order processing efficiency.

Expected Benefits:

- Faster order processing.
- Improved API response times.
- Better reliability during high traffic.
- Improved customer experience.

Current Technical Focus:

The engineering team is currently working on API integration and pe

client_info.txt
Client: Acme Corp

Acme Corp is a retail technology company that is working with our team to improve its online ordering platform.

Industry: Retail Technology

Company Size: Large enterprise client

Primary Contact: Sarah Johnson, Director of Operations

Technical Contact: Michael Chen, Engineering Manager

Current Project: Online Ordering Platform Modernization

Project Goal:
Acme Corp wants to modernize its online ordering platform to improve performan

In [11]:
print(documents.keys())

dict_keys(['company_documents.txt', 'client_info.txt', 'meeting_notes.txt'])


## Step 3 — Text Chunking

Large documents are divided into smaller chunks before generating embeddings.

Chunking makes it possible to retrieve only the relevant portions of a document instead of passing the entire document to the language model.

Each chunk will also store its source document so that we know where the retrieved information came from.

In [12]:
chunk_size = 500
overlap = 50

In [13]:
def create_chunks(text, chunk_size=500, overlap=50):
    chunks = []
    
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        
        chunk = text[start:end]
        
        if chunk.strip():
            chunks.append(chunk)
        
        start += chunk_size - overlap
    
    return chunks

In [14]:
all_chunks = []

for filename, content in documents.items():
    
    document_chunks = create_chunks(
        content,
        chunk_size=500,
        overlap=50
    )
    
    for chunk in document_chunks:
        all_chunks.append({
            "source": filename,
            "text": chunk
        })

print("Total chunks:", len(all_chunks))

Total chunks: 11


In [15]:
for i, chunk in enumerate(all_chunks[:5]):
    print(f"\n--- Chunk {i + 1} ---")
    print("Source:", chunk["source"])
    print("Text:", chunk["text"])


--- Chunk 1 ---
Source: company_documents.txt
Text: Acme Corp — Project Documentation

Project: Online Ordering Platform Modernization

Project Overview:

The project aims to modernize Acme Corp's online ordering platform by improving system performance, API reliability, and order processing efficiency.

Expected Benefits:

- Faster order processing.
- Improved API response times.
- Better reliability during high traffic.
- Improved customer experience.

Current Technical Focus:

The engineering team is currently working on API integration and pe

--- Chunk 2 ---
Source: company_documents.txt
Text: eam is currently working on API integration and performance optimization.

API Integration:

The API integration connects the online ordering platform with Acme Corp's internal order management system.

The integration is currently behind schedule and requires additional testing.

Performance:

Initial testing indicates that order processing time increases during peak traffic periods.

The 

In [16]:
chunk_texts = [
    chunk["text"]
    for chunk in all_chunks
]

print("Number of texts:", len(chunk_texts))

Number of texts: 11


In [17]:
embeddings = embedding_model.encode(
    chunk_texts
)

print("Embedding shape:", embeddings.shape)

Embedding shape: (11, 384)


In [18]:
embedding_matrix = np.array(
    embeddings,
    dtype="float32"
)

print("Embedding matrix shape:", embedding_matrix.shape)

Embedding matrix shape: (11, 384)


In [19]:
faiss.normalize_L2(embedding_matrix)

In [20]:
all_chunks[0]

{'source': 'company_documents.txt',
 'text': "Acme Corp — Project Documentation\n\nProject: Online Ordering Platform Modernization\n\nProject Overview:\n\nThe project aims to modernize Acme Corp's online ordering platform by improving system performance, API reliability, and order processing efficiency.\n\nExpected Benefits:\n\n- Faster order processing.\n- Improved API response times.\n- Better reliability during high traffic.\n- Improved customer experience.\n\nCurrent Technical Focus:\n\nThe engineering team is currently working on API integration and pe"}

In [21]:
print("Total chunks:", len(all_chunks))
print("Embedding shape:", embedding_matrix.shape)

Total chunks: 11
Embedding shape: (11, 384)


## Step 4 — Build the FAISS Vector Store

FAISS is used as the vector database for our RAG system.

The document chunks are converted into embeddings using `all-MiniLM-L6-v2` and stored in a FAISS index.

When the user asks a question, the question will also be converted into an embedding and compared against the stored embeddings to retrieve the most relevant chunks.

In [22]:
dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)

print("FAISS index created!")
print("Vector dimension:", dimension)

FAISS index created!
Vector dimension: 384


In [23]:
index.add(embedding_matrix)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 11


## FAISS Retrieval Function

The retrieval function takes a user query, converts it into an embedding, normalizes it, and searches the FAISS index.

The top `k` most relevant chunks are returned along with their source documents and similarity distances.

In [25]:
def search_documents(query, k=3):
    
    # Convert query into embedding
    query_embedding = embedding_model.encode([query])
    
    # Convert to float32
    query_embedding = np.array(
        query_embedding,
        dtype="float32"
    )
    
    # Normalize query embedding
    faiss.normalize_L2(query_embedding)
    
    # Search FAISS
    distances, indices = index.search(
        query_embedding,
        k
    )
    
    results = []
    
    for distance, idx in zip(distances[0], indices[0]):
        results.append({
            "source": all_chunks[idx]["source"],
            "text": all_chunks[idx]["text"],
            "distance": float(distance)
        })
    
    return results

In [26]:
results = search_documents(
    "What are Acme Corp's main concerns?"
)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", result["source"])
    print("Distance:", result["distance"])
    print("Text:", result["text"])


--- Result 1 ---
Source: client_info.txt
Distance: 0.8681699633598328
Text: Client: Acme Corp

Acme Corp is a retail technology company that is working with our team to improve its online ordering platform.

Industry: Retail Technology

Company Size: Large enterprise client

Primary Contact: Sarah Johnson, Director of Operations

Technical Contact: Michael Chen, Engineering Manager

Current Project: Online Ordering Platform Modernization

Project Goal:
Acme Corp wants to modernize its online ordering platform to improve performance, reliability, and customer experience.

--- Result 2 ---
Source: meeting_notes.txt
Distance: 0.8797493577003479
Text: 
   Owner: Engineering Team
   Status: Pending

Next Meeting Objective:

Review project progress, discuss the revised timeline, and address Acme Corp's concerns regarding API performance and order processing delays.

--- Result 3 ---
Source: meeting_notes.txt
Distance: 1.046472430229187
Text: Previous Meeting Notes — Acme Corp

Meeting Date:

In [27]:
results = search_documents(
    "What action items are still pending?"
)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", result["source"])
    print("Text:", result["text"])


--- Result 1 ---
Source: meeting_notes.txt
Text: 
   Owner: Engineering Team
   Status: Pending

Next Meeting Objective:

Review project progress, discuss the revised timeline, and address Acme Corp's concerns regarding API performance and order processing delays.

--- Result 2 ---
Source: client_info.txt
Text: performance, reliability, and customer experience.

Current Status:
The project is currently in the integration and testing phase.

Key Client Priorities:
1. Improve platform performance.
2. Reduce order processing delays.
3. Improve API reliability.
4. Complete the project within the revised timeline.

Known Client Concerns:
- Delays in API integration.
- Slow order processing during peak hours.
- Concerns about the revised project timeline.

Important Stakeholders:
- Sarah Johnson — Director o

--- Result 3 ---
Source: meeting_notes.txt
Text: Previous Meeting Notes — Acme Corp

Meeting Date: August 20, 2026

Attendees:
- Sarah Johnson — Director of Operations, Acme Corp
- Mic

In [28]:
results = search_documents("What are Acme Corp's main concerns?")

for result in results:
    print(result["source"])
    print(result["text"])

client_info.txt
Client: Acme Corp

Acme Corp is a retail technology company that is working with our team to improve its online ordering platform.

Industry: Retail Technology

Company Size: Large enterprise client

Primary Contact: Sarah Johnson, Director of Operations

Technical Contact: Michael Chen, Engineering Manager

Current Project: Online Ordering Platform Modernization

Project Goal:
Acme Corp wants to modernize its online ordering platform to improve performance, reliability, and customer experience.
meeting_notes.txt

   Owner: Engineering Team
   Status: Pending

Next Meeting Objective:

Review project progress, discuss the revised timeline, and address Acme Corp's concerns regarding API performance and order processing delays.
meeting_notes.txt
Previous Meeting Notes — Acme Corp

Meeting Date: August 20, 2026

Attendees:
- Sarah Johnson — Director of Operations, Acme Corp
- Michael Chen — Engineering Manager, Acme Corp
- David Wilson — Product Manager, Acme Corp
- Project

## Step 5 — Tool 1: Document Search

The Document Search Tool allows the agent to search the client's documents using semantic search.

The tool converts the user's query into an embedding, searches the FAISS vector store, and returns the most relevant document chunks.

This is the first tool available to our agent.

In [29]:
def document_search_tool(query):
    
    results = search_documents(query, k=3)
    
    if not results:
        return "No relevant information found in the documents."
    
    output = []
    
    for i, result in enumerate(results, start=1):
        output.append(
            f"Result {i}\n"
            f"Source: {result['source']}\n"
            f"Content: {result['text']}"
        )
    
    return "\n\n".join(output)

In [30]:
result = document_search_tool(
    "What are Acme Corp's main concerns?"
)

print(result)

Result 1
Source: client_info.txt
Content: Client: Acme Corp

Acme Corp is a retail technology company that is working with our team to improve its online ordering platform.

Industry: Retail Technology

Company Size: Large enterprise client

Primary Contact: Sarah Johnson, Director of Operations

Technical Contact: Michael Chen, Engineering Manager

Current Project: Online Ordering Platform Modernization

Project Goal:
Acme Corp wants to modernize its online ordering platform to improve performance, reliability, and customer experience.

Result 2
Source: meeting_notes.txt
Content: 
   Owner: Engineering Team
   Status: Pending

Next Meeting Objective:

Review project progress, discuss the revised timeline, and address Acme Corp's concerns regarding API performance and order processing delays.

Result 3
Source: meeting_notes.txt
Content: Previous Meeting Notes — Acme Corp

Meeting Date: August 20, 2026

Attendees:
- Sarah Johnson — Director of Operations, Acme Corp
- Michael Chen — Engi

In [31]:
print(
    document_search_tool(
        "What concerns does Acme Corp have?"
    )
)

Result 1
Source: client_info.txt
Content: Client: Acme Corp

Acme Corp is a retail technology company that is working with our team to improve its online ordering platform.

Industry: Retail Technology

Company Size: Large enterprise client

Primary Contact: Sarah Johnson, Director of Operations

Technical Contact: Michael Chen, Engineering Manager

Current Project: Online Ordering Platform Modernization

Project Goal:
Acme Corp wants to modernize its online ordering platform to improve performance, reliability, and customer experience.

Result 2
Source: meeting_notes.txt
Content: 
   Owner: Engineering Team
   Status: Pending

Next Meeting Objective:

Review project progress, discuss the revised timeline, and address Acme Corp's concerns regarding API performance and order processing delays.

Result 3
Source: meeting_notes.txt
Content: Previous Meeting Notes — Acme Corp

Meeting Date: August 20, 2026

Attendees:
- Sarah Johnson — Director of Operations, Acme Corp
- Michael Chen — Engi

In [32]:
print(
    document_search_tool(
        "What is the current status of the project?"
    )
)

Result 1
Source: client_info.txt
Content: performance, reliability, and customer experience.

Current Status:
The project is currently in the integration and testing phase.

Key Client Priorities:
1. Improve platform performance.
2. Reduce order processing delays.
3. Improve API reliability.
4. Complete the project within the revised timeline.

Known Client Concerns:
- Delays in API integration.
- Slow order processing during peak hours.
- Concerns about the revised project timeline.

Important Stakeholders:
- Sarah Johnson — Director o

Result 2
Source: company_documents.txt
Content: ear communication about project progress, delivery timelines, and technical issues.

The client has requested regular updates regarding API performance and project milestones.

Result 3
Source: meeting_notes.txt
Content: ce during peak hours needs improvement.
4. A revised project timeline needs to be shared with the client.

Open Action Items:

1. Engineering team to complete API integration.
   Owner: E

In [33]:
print(
    document_search_tool(
        "What technical problems are being investigated?"
    )
)

Result 1
Source: client_info.txt
Content: performance, reliability, and customer experience.

Current Status:
The project is currently in the integration and testing phase.

Key Client Priorities:
1. Improve platform performance.
2. Reduce order processing delays.
3. Improve API reliability.
4. Complete the project within the revised timeline.

Known Client Concerns:
- Delays in API integration.
- Slow order processing during peak hours.
- Concerns about the revised project timeline.

Important Stakeholders:
- Sarah Johnson — Director o

Result 2
Source: company_documents.txt
Content: I response times to identify performance bottlenecks.

Testing:

Performance testing must be completed before the next production release.

The QA team is responsible for validating system performance under normal and peak traffic conditions.

Release Planning:

The original release timeline has been revised because of the API integration delay.

A new project timeline needs to be shared with Acme Corp.



## Step 6 — Tool 2: Meeting Notes Retrieval

The Meeting Notes Tool retrieves information from previous meetings with the client.

This tool helps the agent identify previous discussion points, client concerns, open action items, and the objectives of the next meeting.

The tool provides the agent with a focused source of information instead of requiring it to search the entire knowledge base.

In [34]:
meeting_notes = documents["meeting_notes.txt"]

print(meeting_notes[:1000])

Previous Meeting Notes — Acme Corp

Meeting Date: August 20, 2026

Attendees:
- Sarah Johnson — Director of Operations, Acme Corp
- Michael Chen — Engineering Manager, Acme Corp
- David Wilson — Product Manager, Acme Corp
- Project Team

Meeting Summary:

The team discussed the current progress of the online ordering platform modernization project.

Acme Corp raised concerns about delays in the API integration.

The client also reported that order processing becomes slower during peak traffic periods.

The project team explained that additional performance testing is required before the next release.

Sarah Johnson requested a revised project timeline.

Michael Chen requested additional information about the API performance test results.

Key Discussion Points:

1. API integration is behind the original schedule.
2. Performance testing needs to be completed.
3. Order processing performance during peak hours needs improvement.
4. A revised project timeline needs to be shared with the cl

In [39]:
def get_meeting_notes(request):
    
    request = request.lower()
    
    if "action" in request or "pending" in request:
        start_marker = "Open Action Items:"
        end_marker = "Next Meeting Objective:"
        
    elif "concern" in request or "issue" in request:
        start_marker = "Key Discussion Points:"
        end_marker = "Open Action Items:"
        
    elif "next" in request:
        start_marker = "Next Meeting Objective:"
        end_marker = None
        
    else:
        return meeting_notes
    
    start = meeting_notes.find(start_marker)
    
    if start == -1:
        return "No relevant meeting information found."
    
    start = start + len(start_marker)
    
    if end_marker:
        end = meeting_notes.find(end_marker, start)
        
        if end == -1:
            end = len(meeting_notes)
    else:
        end = len(meeting_notes)
    
    result = meeting_notes[start:end].strip()
    
    return f"{start_marker}\n\n{result}"

In [40]:
result = get_meeting_notes(
    "What are the pending action items?"
)

print(result)

Open Action Items:

1. Engineering team to complete API integration.
   Owner: Engineering Team
   Status: In Progress

2. Complete performance testing.
   Owner: QA Team
   Status: Pending

3. Share revised project timeline with Acme Corp.
   Owner: Project Manager
   Status: Pending

4. Share API performance test results with Michael Chen.
   Owner: Engineering Team
   Status: Pending


In [41]:
result = get_meeting_notes(
    "What concerns were discussed in the previous meeting?"
)

print(result)

Key Discussion Points:

1. API integration is behind the original schedule.
2. Performance testing needs to be completed.
3. Order processing performance during peak hours needs improvement.
4. A revised project timeline needs to be shared with the client.


In [42]:
result = get_meeting_notes(
    "What should be discussed in the next meeting?"
)

print(result)

Next Meeting Objective:

Review project progress, discuss the revised timeline, and address Acme Corp's concerns regarding API performance and order processing delays.


In [43]:
print("TOOL 1 — DOCUMENT SEARCH")
print("=" * 50)

print(
    document_search_tool(
        "What are Acme Corp's main concerns?"
    )
)

print("\n\nTOOL 2 — MEETING NOTES")
print("=" * 50)

print(
    get_meeting_notes(
        "What are the pending action items?"
    )
)

TOOL 1 — DOCUMENT SEARCH
Result 1
Source: client_info.txt
Content: Client: Acme Corp

Acme Corp is a retail technology company that is working with our team to improve its online ordering platform.

Industry: Retail Technology

Company Size: Large enterprise client

Primary Contact: Sarah Johnson, Director of Operations

Technical Contact: Michael Chen, Engineering Manager

Current Project: Online Ordering Platform Modernization

Project Goal:
Acme Corp wants to modernize its online ordering platform to improve performance, reliability, and customer experience.

Result 2
Source: meeting_notes.txt
Content: 
   Owner: Engineering Team
   Status: Pending

Next Meeting Objective:

Review project progress, discuss the revised timeline, and address Acme Corp's concerns regarding API performance and order processing delays.

Result 3
Source: meeting_notes.txt
Content: Previous Meeting Notes — Acme Corp

Meeting Date: August 20, 2026

Attendees:
- Sarah Johnson — Director of Operations, Acme C

## Step 7 — Short-Term Memory

Short-term memory allows the assistant to remember information from the current conversation.

It stores the recent user questions and assistant responses so that follow-up questions can be understood in context.

For example, if the user first asks about Acme Corp and then asks "What should I discuss with them?", the assistant can use the previous conversation to understand who "them" refers to.

In [44]:
conversation_history = []

In [45]:
def add_to_memory(role, message):
    conversation_history.append({
        "role": role,
        "message": message
    })

In [46]:
def get_short_term_memory():
    
    if not conversation_history:
        return "No conversation history available."
    
    memory = []
    
    for item in conversation_history:
        memory.append(
            f"{item['role'].capitalize()}: {item['message']}"
        )
    
    return "\n".join(memory)

In [47]:
add_to_memory(
    "user",
    "Prepare me for my meeting with Acme Corp."
)

In [48]:
add_to_memory(
    "assistant",
    "Acme Corp is currently concerned about API integration delays and order processing performance."
)

In [49]:
add_to_memory(
    "user",
    "What should I discuss with them?"
)

In [50]:
print(get_short_term_memory())

User: Prepare me for my meeting with Acme Corp.
Assistant: Acme Corp is currently concerned about API integration delays and order processing performance.
User: What should I discuss with them?


In [51]:
def get_recent_context(max_messages=5):
    
    recent_messages = conversation_history[-max_messages:]
    
    context = []
    
    for item in recent_messages:
        context.append(
            f"{item['role'].capitalize()}: {item['message']}"
        )
    
    return "\n".join(context)

In [52]:
print(get_recent_context())

User: Prepare me for my meeting with Acme Corp.
Assistant: Acme Corp is currently concerned about API integration delays and order processing performance.
User: What should I discuss with them?


## Step 8 — Long-Term Memory

Long-term memory stores important information that should be available across different conversations or sessions.

In this project, long-term memory is implemented using FAISS. Each memory is converted into an embedding using `all-MiniLM-L6-v2` and stored in a FAISS vector index.

When the agent receives a new request, it can search the stored memories and retrieve information that is relevant to the current conversation.

In [53]:
long_term_memories = []

In [54]:
memory_dimension = 384

memory_index = faiss.IndexFlatL2(memory_dimension)

print("Long-term memory index created.")

Long-term memory index created.


In [55]:
def store_memory(memory):
    
    # Convert memory into embedding
    memory_embedding = embedding_model.encode([memory])
    
    # Convert to float32
    memory_embedding = np.array(
        memory_embedding,
        dtype="float32"
    )
    
    # Normalize embedding
    faiss.normalize_L2(memory_embedding)
    
    # Add embedding to FAISS
    memory_index.add(memory_embedding)
    
    # Store original memory text
    long_term_memories.append(memory)
    
    print("Memory stored successfully.")

In [56]:
store_memory(
    "Acme Corp's main concern is the delay in API integration."
)

Memory stored successfully.


In [57]:
store_memory(
    "Acme Corp wants regular updates about project progress and delivery timelines."
)

Memory stored successfully.


In [58]:
store_memory(
    "Sarah Johnson is the Director of Operations at Acme Corp."
)

Memory stored successfully.


In [59]:
store_memory(
    "Michael Chen is the Engineering Manager at Acme Corp."
)

Memory stored successfully.


In [60]:
print("Total stored memories:", memory_index.ntotal)

Total stored memories: 4


In [61]:
def retrieve_memory(query, k=3):
    
    if memory_index.ntotal == 0:
        return "No long-term memories available."
    
    # Convert query into embedding
    query_embedding = embedding_model.encode([query])
    
    # Convert to float32
    query_embedding = np.array(
        query_embedding,
        dtype="float32"
    )
    
    # Normalize query
    faiss.normalize_L2(query_embedding)
    
    # Search memory index
    k = min(k, memory_index.ntotal)
    
    distances, indices = memory_index.search(
        query_embedding,
        k
    )
    
    results = []
    
    for distance, idx in zip(distances[0], indices[0]):
        results.append({
            "memory": long_term_memories[idx],
            "distance": float(distance)
        })
    
    return results

In [62]:
results = retrieve_memory(
    "What is Acme Corp concerned about?"
)

for i, result in enumerate(results, start=1):
    print(f"\n--- Memory {i} ---")
    print("Distance:", result["distance"])
    print("Memory:", result["memory"])


--- Memory 1 ---
Distance: 0.6279377937316895
Memory: Acme Corp's main concern is the delay in API integration.

--- Memory 2 ---
Distance: 0.7128497958183289
Memory: Sarah Johnson is the Director of Operations at Acme Corp.

--- Memory 3 ---
Distance: 0.9351022839546204
Memory: Michael Chen is the Engineering Manager at Acme Corp.


In [63]:
results = retrieve_memory(
    "Who is the main contact at Acme Corp?"
)

for i, result in enumerate(results, start=1):
    print(f"\n--- Memory {i} ---")
    print("Memory:", result["memory"])


--- Memory 1 ---
Memory: Sarah Johnson is the Director of Operations at Acme Corp.

--- Memory 2 ---
Memory: Michael Chen is the Engineering Manager at Acme Corp.

--- Memory 3 ---
Memory: Acme Corp's main concern is the delay in API integration.


In [64]:
def memory_retrieval_tool(query):
    
    results = retrieve_memory(query, k=3)
    
    if isinstance(results, str):
        return results
    
    output = []
    
    for i, result in enumerate(results, start=1):
        output.append(
            f"Memory {i}\n"
            f"{result['memory']}"
        )
    
    return "\n\n".join(output)

In [65]:
print(
    memory_retrieval_tool(
        "What does Acme Corp care about?"
    )
)

Memory 1
Acme Corp's main concern is the delay in API integration.

Memory 2
Sarah Johnson is the Director of Operations at Acme Corp.

Memory 3
Acme Corp wants regular updates about project progress and delivery timelines.


## Step 9 — Build the Agentic Workflow

The agentic workflow allows the AI system to decide which tool should be used to answer a user's request.

Instead of directly generating an answer, the agent can:

1. Understand the user's request.
2. Decide which information is required.
3. Select the appropriate tool.
4. Retrieve relevant information.
5. Use the retrieved information to generate a final response.

The agent will have access to three tools:

- Document Search Tool
- Meeting Notes Tool
- Memory Retrieval Tool

In [67]:
tools = {
    "document_search": document_search_tool,
    "meeting_notes": get_meeting_notes,
    "memory_retrieval": memory_retrieval_tool
}

print("Available tools:")

for tool_name in tools:
    print("-", tool_name)

Available tools:
- document_search
- meeting_notes
- memory_retrieval


In [68]:
tool_descriptions = """
You are an AI Meeting Preparation Agent.

You have access to the following tools:

1. document_search
Use this tool to search Acme Corp client information, project
documentation, technical information, requirements, stakeholders,
and other general client information.

2. meeting_notes
Use this tool to retrieve information from previous meetings,
including discussion points, client concerns, open action items,
and next meeting objectives.

3. memory_retrieval
Use this tool to retrieve information remembered from previous
sessions, such as important client preferences, previously
identified concerns, and important contacts.

Choose the appropriate tool based on the user's request.
"""

In [69]:
def choose_tool(question):
    
    prompt = f"""
{tool_descriptions}

User request:
{question}

Decide which tool should be used.

Return ONLY one of these exact values:

document_search
meeting_notes
memory_retrieval

If multiple tools are required, return:
multiple
"""
    
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    
    return response.text.strip()

In [70]:
question = "What are Acme Corp's main technical issues?"

tool = choose_tool(question)

print("Selected tool:", tool)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Selected tool: document_search


In [71]:
question = "What action items are still pending from the previous meeting?"

tool = choose_tool(question)

print("Selected tool:", tool)

Selected tool: meeting_notes


In [72]:
question = "What concerns has Acme previously mentioned?"

tool = choose_tool(question)

print("Selected tool:", tool)

Selected tool: multiple


In [73]:
print(choose_tool(
    "What information do we remember about Acme?"
))

memory_retrieval


## Step 10 — Execute Tools and Generate the Agent Response

The agent now connects its decision-making process with the available tools.

Based on the user's request, the agent selects the appropriate tool, executes it, retrieves the relevant information, and uses that information to generate a final response.

This demonstrates an agentic workflow because the system is not simply answering the question directly. It first determines what information is required and then uses an appropriate tool to obtain that information.

In [80]:
def execute_tool(tool_name, query):
    
    if tool_name == "document_search":
        return document_search_tool(query)
    
    elif tool_name == "meeting_notes":
        return get_meeting_notes(query)
    
    elif tool_name == "memory_retrieval":
        return memory_retrieval_tool(query)
    
    else:
        return "No valid tool selected."

## Step 11 — Multi-Tool Agent

Some user requests require information from multiple sources.

For example, preparing for a client meeting may require:

- Client information from the document search tool
- Previous meeting information from the meeting notes tool
- Previously stored information from long-term memory

The agent can therefore use multiple tools, combine their results, and provide a single final response.

In [81]:
def run_agent(question):
    
    # Step 1: Ask Gemini which tool(s) are required
    selected_tool = choose_tool(question)
    
    print("Selected tool:", selected_tool)
    
    # Step 2: Execute the required tool(s)
    
    if selected_tool == "document_search":
        
        tool_result = execute_tool(
            "document_search",
            question
        )
    
    elif selected_tool == "meeting_notes":
        
        tool_result = execute_tool(
            "meeting_notes",
            question
        )
    
    elif selected_tool == "memory_retrieval":
        
        tool_result = execute_tool(
            "memory_retrieval",
            question
        )
    
    elif selected_tool == "multiple":
        
        document_result = execute_tool(
            "document_search",
            question
        )
        
        meeting_result = execute_tool(
            "meeting_notes",
            question
        )
        
        memory_result = execute_tool(
            "memory_retrieval",
            question
        )
        
        tool_result = f"""
DOCUMENT SEARCH RESULTS:
{document_result}

MEETING NOTES RESULTS:
{meeting_result}

LONG-TERM MEMORY RESULTS:
{memory_result}
"""
    
    else:
        return "Unable to determine the appropriate tool."
    
    
    # Step 3: Generate final answer using retrieved information
    
    prompt = f"""
You are an AI Meeting Preparation Assistant.

Answer the user's question using the retrieved information below.

Do not invent information that is not present in the retrieved data.

If the information is insufficient, clearly say that the
information is not available.

Retrieved Information:
{tool_result}

User Question:
{question}

Provide a clear and concise answer.
"""
    
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    
    return response.text

In [82]:
answer = run_agent(
    "Prepare me for my meeting with Acme Corp."
)

print("\nFinal Answer:")
print(answer)

Selected tool: multiple

Final Answer:
Here is your meeting preparation brief for the upcoming meeting with **Acme Corp**:

---

### **Client & Project Context**
* **Client:** Acme Corp (Retail Technology)
* **Current Project:** Online Ordering Platform Modernization
* **Project Goal:** Improve performance, reliability, and customer experience of the online ordering platform.

---

### **Key Stakeholders / Attendees**
* **Sarah Johnson** — Director of Operations (Primary Contact)
* **Michael Chen** — Engineering Manager (Technical Contact)
* **David Wilson** — Product Manager
* **Project Team**

---

### **Next Meeting Objective**
Review project progress, discuss the revised timeline, and address Acme Corp's concerns regarding API performance and order processing delays.

---

### **Key Discussion Points & Client Concerns**
1. **API Integration Delays:** API integration is behind the original schedule.
2. **Order Processing Delays:** Order processing slows down during peak traffic peri

## Step 12 — Integrate Short-Term Memory

The agent now automatically stores the user's questions and the assistant's responses in short-term memory.

This allows the agent to understand follow-up questions based on the current conversation context.

For example:

User: Prepare me for my meeting with Acme Corp.

Assistant: Acme Corp is concerned about API delays.

User: What should I discuss with them?

The agent can use the previous conversation to understand that "them" refers to Acme Corp.

In [83]:
conversation_history = []

print("Short-term memory cleared.")

Short-term memory cleared.


In [84]:
def run_agent(question):
    
    # Step 1: Add the user's question to short-term memory
    add_to_memory("user", question)
    
    # Step 2: Get recent conversation context
    conversation_context = get_recent_context()
    
    # Step 3: Ask Gemini which tool(s) are required
    tool_selection_prompt = f"""
You are an AI Meeting Preparation Agent.

Available tools:

1. document_search
Use this for client information, project information,
technical issues, requirements, and general documents.

2. meeting_notes
Use this for previous meeting discussions, concerns,
action items, and meeting objectives.

3. memory_retrieval
Use this for information stored from previous sessions.

Conversation History:
{conversation_context}

Current User Question:
{question}

Decide which tool should be used.

Return ONLY one of:

document_search
meeting_notes
memory_retrieval
multiple
"""
    
    tool_response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=tool_selection_prompt
    )
    
    selected_tool = tool_response.text.strip()
    
    print("Selected tool:", selected_tool)
    
    
    # Step 4: Execute the selected tool(s)
    
    if selected_tool == "document_search":
        
        tool_result = execute_tool(
            "document_search",
            question
        )
    
    elif selected_tool == "meeting_notes":
        
        tool_result = execute_tool(
            "meeting_notes",
            question
        )
    
    elif selected_tool == "memory_retrieval":
        
        tool_result = execute_tool(
            "memory_retrieval",
            question
        )
    
    elif selected_tool == "multiple":
        
        document_result = execute_tool(
            "document_search",
            question
        )
        
        meeting_result = execute_tool(
            "meeting_notes",
            question
        )
        
        memory_result = execute_tool(
            "memory_retrieval",
            question
        )
        
        tool_result = f"""
DOCUMENT SEARCH RESULTS:
{document_result}

MEETING NOTES RESULTS:
{meeting_result}

LONG-TERM MEMORY RESULTS:
{memory_result}
"""
    
    else:
        tool_result = "No relevant tool was selected."
    
    
    # Step 5: Generate the final answer
    
    final_prompt = f"""
You are an AI Meeting Preparation Assistant.

Use the conversation history and retrieved information
to answer the user's question.

Do not invent information.

Conversation History:
{conversation_context}

Retrieved Information:
{tool_result}

Current User Question:
{question}

Provide a clear and concise answer.
"""
    
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=final_prompt
    )
    
    answer = response.text
    
    
    # Step 6: Store the assistant's response
    add_to_memory("assistant", answer)
    
    return answer

In [85]:
answer = run_agent(
    "Prepare me for my meeting with Acme Corp."
)

print("\nAssistant:")
print(answer)

Selected tool: multiple

Assistant:
Here is your preparation summary for the upcoming meeting with Acme Corp:

---

### **Meeting Overview & Objective**
* **Project:** Online Ordering Platform Modernization (Goal: Improve performance, reliability, and customer experience).
* **Objective:** Review project progress, present the revised project timeline, and address Acme Corp's concerns regarding API performance and order processing delays during peak traffic.

---

### **Key Attendees (Acme Corp)**
* **Sarah Johnson** – Director of Operations (Primary Contact)
* **Michael Chen** – Engineering Manager (Technical Contact)
* **David Wilson** – Product Manager

---

### **Key Issues & Client Concerns to Address**
1. **API Integration Delays:** The API integration is behind schedule.
2. **Order Processing Performance:** Order processing slows down during peak traffic periods.
3. **Performance Testing:** Explain the status of performance testing required before the next release.

---

### **Pe

In [86]:
answer = run_agent(
    "What should I discuss with them?"
)

print("\nAssistant:")
print(answer)

Selected tool: meeting_notes

Assistant:
Based on your previous meeting and notes, you should discuss the following key topics with Acme Corp:

1. **Project Progress & Overview:** Provide an update on the overall status of the Online Ordering Platform Modernization project.
2. **Revised Project Timeline:** Present the updated project timeline (requested by Sarah Johnson).
3. **API Integration & Performance:** 
   * Update them on the completion status of the API integration, which was behind schedule.
   * Share the API performance test results (requested by Michael Chen).
4. **Order Processing Delays:** Address their concerns and present updates on improving order processing performance during peak traffic periods.
5. **Performance Testing:** Update them on the status of the QA performance testing required before the next release.


In [87]:
print(get_short_term_memory())

User: Prepare me for my meeting with Acme Corp.
Assistant: Here is your preparation summary for the upcoming meeting with Acme Corp:

---

### **Meeting Overview & Objective**
* **Project:** Online Ordering Platform Modernization (Goal: Improve performance, reliability, and customer experience).
* **Objective:** Review project progress, present the revised project timeline, and address Acme Corp's concerns regarding API performance and order processing delays during peak traffic.

---

### **Key Attendees (Acme Corp)**
* **Sarah Johnson** – Director of Operations (Primary Contact)
* **Michael Chen** – Engineering Manager (Technical Contact)
* **David Wilson** – Product Manager

---

### **Key Issues & Client Concerns to Address**
1. **API Integration Delays:** The API integration is behind schedule.
2. **Order Processing Performance:** Order processing slows down during peak traffic periods.
3. **Performance Testing:** Explain the status of performance testing required before the next 

In [88]:
answer = run_agent(
    "What action items are still pending?"
)

print("\nAssistant:")
print(answer)

Selected tool: meeting_notes

Assistant:
Here are the open action items for your meeting:

1. **Share revised project timeline with Acme Corp** 
   * **Owner:** Project Manager
   * **Status:** Pending

2. **Share API performance test results with Michael Chen** 
   * **Owner:** Engineering Team
   * **Status:** Pending

3. **Complete performance testing** 
   * **Owner:** QA Team
   * **Status:** Pending

4. **Complete API integration** 
   * **Owner:** Engineering Team
   * **Status:** In Progress


## Step 13 — Persistent Long-Term Memory

Long-term memory should remain available even after the current session ends.

In this project, FAISS stores the embeddings of long-term memories, while a JSON file stores the corresponding memory text.

The FAISS index and JSON file are saved to disk and can be loaded again when the application starts.

This allows the agent to retrieve relevant information across different sessions.

In [89]:
import os

os.makedirs("memory", exist_ok=True)

print("Memory folder ready.")

Memory folder ready.


In [90]:
import json

def save_long_term_memory():
    
    # Save FAISS index
    faiss.write_index(
        memory_index,
        "memory/long_term_memory.index"
    )
    
    # Save original memory text
    with open(
        "memory/long_term_memories.json",
        "w"
    ) as file:
        
        json.dump(
            long_term_memories,
            file,
            indent=4
        )
    
    print("Long-term memory saved successfully.")

In [91]:
save_long_term_memory()

Long-term memory saved successfully.


In [92]:
def load_long_term_memory():
    
    global memory_index
    global long_term_memories
    
    if os.path.exists("memory/long_term_memory.index"):
        
        memory_index = faiss.read_index(
            "memory/long_term_memory.index"
        )
        
    else:
        
        memory_index = faiss.IndexFlatL2(384)
    
    
    if os.path.exists("memory/long_term_memories.json"):
        
        with open(
            "memory/long_term_memories.json",
            "r"
        ) as file:
            
            long_term_memories = json.load(file)
            
    else:
        
        long_term_memories = []
    
    
    print(
        "Long-term memories loaded:",
        len(long_term_memories)
    )

In [93]:
load_long_term_memory()

Long-term memories loaded: 4


In [94]:
print(long_term_memories)

["Acme Corp's main concern is the delay in API integration.", 'Acme Corp wants regular updates about project progress and delivery timelines.', 'Sarah Johnson is the Director of Operations at Acme Corp.', 'Michael Chen is the Engineering Manager at Acme Corp.']


In [95]:
print(
    memory_retrieval_tool(
        "What is Acme Corp concerned about?"
    )
)

Memory 1
Acme Corp's main concern is the delay in API integration.

Memory 2
Sarah Johnson is the Director of Operations at Acme Corp.

Memory 3
Michael Chen is the Engineering Manager at Acme Corp.


In [96]:
long_term_memories = []

memory_index = faiss.IndexFlatL2(384)

print("Current memory cleared.")

Current memory cleared.


In [97]:
print("Memories:", len(long_term_memories))
print("Vectors:", memory_index.ntotal)

Memories: 0
Vectors: 0


In [98]:
load_long_term_memory()

Long-term memories loaded: 4


In [99]:
def store_memory(memory):
    
    # Convert memory into embedding
    memory_embedding = embedding_model.encode([memory])
    
    memory_embedding = np.array(
        memory_embedding,
        dtype="float32"
    )
    
    # Normalize embedding
    faiss.normalize_L2(memory_embedding)
    
    # Add embedding to FAISS
    memory_index.add(memory_embedding)
    
    # Store original memory
    long_term_memories.append(memory)
    
    # Save to disk
    save_long_term_memory()
    
    print("Memory stored successfully.")

In [100]:
store_memory(
    "Acme Corp prefers weekly project status updates."
)

Long-term memory saved successfully.
Memory stored successfully.


In [101]:
save_long_term_memory()

Long-term memory saved successfully.


In [102]:
load_long_term_memory()

Long-term memories loaded: 5


In [103]:
print(
    memory_retrieval_tool(
        "What is Acme Corp concerned about?"
    )
)

Memory 1
Acme Corp's main concern is the delay in API integration.

Memory 2
Sarah Johnson is the Director of Operations at Acme Corp.

Memory 3
Michael Chen is the Engineering Manager at Acme Corp.


## Step 14 — Meeting Preparation Brief

The Meeting Preparation Brief combines information retrieved from multiple sources and presents it in a concise format for the upcoming client meeting.

The agent retrieves:

- Client and project information
- Previous meeting discussions
- Open action items
- Relevant long-term memories

The retrieved information is then provided to Gemini, which generates a structured meeting brief containing key concerns, talking points, action items, and recommended next steps.

In [116]:
def generate_meeting_brief(client_name):
    
    # Retrieve client information
    document_info = document_search_tool(
        f"Provide important information about {client_name}, "
        f"including project status, client concerns, requirements, "
        f"stakeholders and technical issues."
    )
    
    # Retrieve previous meeting information
    meeting_info = get_meeting_notes(
        "What were the important discussion points, concerns, "
        "and action items from the previous meeting?"
    )
    
    # Retrieve long-term memories
    memory_info = memory_retrieval_tool(
        f"What important information do we remember about {client_name}?"
    )
    
    # Combine all retrieved information
    context = f"""
CLIENT / PROJECT INFORMATION:
{document_info}

PREVIOUS MEETING INFORMATION:
{meeting_info}

LONG-TERM MEMORY:
{memory_info}
"""
    
    # Generate meeting brief
    prompt = f"""
You are an AI Meeting Preparation Assistant.

Prepare a concise meeting brief for an upcoming meeting
with {client_name}.

Use ONLY the information provided below.

Do not invent facts.

Organize the response using these sections:

1. Client Overview
2. Key Concerns
3. Previous Meeting Discussion
4. Open Action Items
5. Important Contacts
6. Recommended Talking Points
7. Next Steps

Retrieved Information:
{context}

Make the brief clear, concise, and useful for a manager
who has only a few minutes to prepare.
"""
    
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    
    return response.text

In [117]:
meeting_brief = generate_meeting_brief(
    "Acme Corp"
)

print(meeting_brief)

Here is your concise meeting brief for the upcoming meeting with Acme Corp:

---

### 1. Client Overview
* **Client:** Acme Corp
* **Industry & Size:** Retail Technology | Large enterprise client
* **Current Project:** Online Ordering Platform Modernization
* **Project Goal:** Modernize the platform to improve system performance, API reliability, order processing efficiency, stability during high traffic, and customer experience.
* **Current Focus:** API integration and performance testing.

### 2. Key Concerns
* **Main Concern:** Delay in API integration.
* **Additional Concerns:** API performance and order processing delays.

### 3. Previous Meeting Discussion
* Context and past discussions focused on addressing API integration progress, order processing delays, performance testing requirements, and updating the overall project timeline.

### 4. Open Action Items
1. **Complete API integration**  
   * *Owner:* Engineering Team | *Status:* In Progress
2. **Complete performance testing

In [118]:
document_info = document_search_tool(
    "Provide important information about Acme Corp"
)

meeting_info = get_meeting_notes(
    "What were the previous discussion points and action items?"
)

memory_info = memory_retrieval_tool(
    "What important information do we remember about Acme Corp?"
)

print("===== DOCUMENT INFORMATION =====")
print(document_info)

print("\n===== MEETING NOTES =====")
print(meeting_info)

print("\n===== LONG-TERM MEMORY =====")
print(memory_info)

===== DOCUMENT INFORMATION =====
Result 1
Source: client_info.txt
Content: Client: Acme Corp

Acme Corp is a retail technology company that is working with our team to improve its online ordering platform.

Industry: Retail Technology

Company Size: Large enterprise client

Primary Contact: Sarah Johnson, Director of Operations

Technical Contact: Michael Chen, Engineering Manager

Current Project: Online Ordering Platform Modernization

Project Goal:
Acme Corp wants to modernize its online ordering platform to improve performance, reliability, and customer experience.

Result 2
Source: meeting_notes.txt
Content: 
   Owner: Engineering Team
   Status: Pending

Next Meeting Objective:

Review project progress, discuss the revised timeline, and address Acme Corp's concerns regarding API performance and order processing delays.

Result 3
Source: meeting_notes.txt
Content: Previous Meeting Notes — Acme Corp

Meeting Date: August 20, 2026

Attendees:
- Sarah Johnson — Director of Operations

## Step 15 — Structured Meeting Brief

The retrieved information is organized into a structured meeting brief.

The brief combines information from document search, previous meeting notes, and long-term memory.

This provides the manager with the most important information needed to prepare for the client meeting.

In [119]:
meeting_brief = f"""
========================================
       ACME CORP MEETING BRIEF
========================================

CLIENT / PROJECT INFORMATION
----------------------------
{document_info}


PREVIOUS MEETING INFORMATION
----------------------------
{meeting_info}


LONG-TERM MEMORY
----------------
{memory_info}


========================================
"""
    
print(meeting_brief)


       ACME CORP MEETING BRIEF

CLIENT / PROJECT INFORMATION
----------------------------
Result 1
Source: client_info.txt
Content: Client: Acme Corp

Acme Corp is a retail technology company that is working with our team to improve its online ordering platform.

Industry: Retail Technology

Company Size: Large enterprise client

Primary Contact: Sarah Johnson, Director of Operations

Technical Contact: Michael Chen, Engineering Manager

Current Project: Online Ordering Platform Modernization

Project Goal:
Acme Corp wants to modernize its online ordering platform to improve performance, reliability, and customer experience.

Result 2
Source: meeting_notes.txt
Content: 
   Owner: Engineering Team
   Status: Pending

Next Meeting Objective:

Review project progress, discuss the revised timeline, and address Acme Corp's concerns regarding API performance and order processing delays.

Result 3
Source: meeting_notes.txt
Content: Previous Meeting Notes — Acme Corp

Meeting Date: August 20,